### Ноутбук, в котором генерируются фичи с использованием OpenFE

In [2]:
import pandas as pd

from ml_pipeline.classification.config import classification_config as config
from ml_pipeline.base.utils import set_seed

In [3]:
set_seed(config.general.seed)

In [4]:
full_df = pd.read_csv(config.paths.train, index_col="PassengerId")
full_test_df = pd.read_csv(config.paths.test, index_col="PassengerId")
full_df.head()

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
PassengerId,,,,,,,,,,,
1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


Логистическая регрессия - просто как модель, из параметров которой можно взять список препроцессоров

По сути тут используется конфиг пайплайна без самого пайплайна

In [5]:
config_logreg = config.copy()

model_name = "linear"

config_logreg.experiment.to_train = [
    {
        "model": model_name
    }
]

config_logreg.split.cv = False

In [6]:
from ml_pipeline.base.utils import build_transformers, apply_transformers
from ml_pipeline.classification.pipeline import ClassificationPipeline

pipeline = ClassificationPipeline()

pipeline.preprocessor_registry

model_config = config_logreg.models.get(model_name)

target = config_logreg.data.target_col

y = full_df[target].values
X_train_raw = full_df.drop(columns=[target])

preprocess_transformers = build_transformers(
    preprocessor_registry=pipeline.preprocessor_registry,
    model_config=model_config,
    config=config_logreg,
)

X_train = apply_transformers(preprocess_transformers, X_train_raw, fit=True)
X_test = apply_transformers(preprocess_transformers, full_test_df, fit=False)

In [7]:
from openfe import OpenFE, transform

ofe = OpenFE()

features = ofe.fit(data=X_train, label=y, n_jobs=4)

The number of candidate features is 1712
Start stage I selection.


100%|██████████| 16/16 [00:05<00:00,  3.07it/s]


1373 same features have been deleted.
Meet early-stopping in successive feature-wise halving.


100%|██████████| 16/16 [00:03<00:00,  4.53it/s]


The number of remaining candidate features is 337
Start stage II selection.


100%|██████████| 16/16 [00:02<00:00,  6.28it/s]


Finish data processing.
[LightGBM] [Info] Number of positive: 273, number of negative: 439
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002234 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 8268
[LightGBM] [Info] Number of data points in the train set: 712, number of used features: 347


In [8]:
ofe_train_x, ofe_test_x = transform(X_train, X_test, features, n_jobs=4) # transform the train and test data according to generated features.

Поменять название индекса и влить таргет в один датафрейм

In [9]:
ofe_train_x.index.name = 'PassengerId'
ofe_train_x['Survived'] = y
ofe_train_x

C:\Users\az\AppData\Local\Temp\ipykernel_8488\1175043723.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  ofe_train_x['Survived'] = y


,Pclass,Sex,SibSp,Parch,Family_Size,Alone,More_Than_4_relatives,Ticket_Type,Embarked_C,Embarked_Q,...,autoFE_f_328,autoFE_f_329,autoFE_f_330,autoFE_f_331,autoFE_f_332,autoFE_f_333,autoFE_f_334,autoFE_f_335,autoFE_f_336,Survived
PassengerId,,,,,,,,,,,,,,,,,,,,,
1,3,0,1,0,1,0,0,0.810458,0.0,0.0,...,NaN,0.0,1.0,845.0,1.0,2.0,0.0,0.420000,0.295652,0
2,1,1,1,0,1,0,0,0.895425,1.0,0.0,...,NaN,0.5,1.0,382.0,2.0,0.0,1.0,0.883721,0.867865,1
3,3,1,0,0,0,1,0,0.967320,0.0,0.0,...,1.0,1.0,0.0,845.0,0.0,3.0,NaN,0.420000,0.795652,1
4,1,1,1,0,1,0,0,0.019608,0.0,0.0,...,NaN,0.5,1.0,845.0,2.0,0.0,1.0,0.383721,0.867865,1
5,3,0,0,0,0,1,0,0.633987,0.0,0.0,...,0.0,0.0,0.0,845.0,0.0,3.0,NaN,0.383721,0.367865,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
887,2,0,0,0,0,1,0,0.150327,0.0,0.0,...,0.0,0.0,0.0,845.0,0.0,2.0,NaN,0.420000,0.295652,0
888,1,1,0,0,0,1,0,0.013072,0.0,0.0,...,1.0,1.0,0.0,845.0,0.0,1.0,NaN,0.420000,0.795652,1
889,3,1,1,2,3,0,0,0.980392,0.0,0.0,...,NaN,1.0,5.0,845.0,3.0,0.0,1.0,0.420000,0.795652,0


In [10]:
ofe_test_x

,Pclass,Sex,SibSp,Parch,Family_Size,Alone,More_Than_4_relatives,Ticket_Type,Embarked_C,Embarked_Q,...,autoFE_f_327,autoFE_f_328,autoFE_f_329,autoFE_f_330,autoFE_f_331,autoFE_f_332,autoFE_f_333,autoFE_f_334,autoFE_f_335,autoFE_f_336
PassengerId,,,,,,,,,,,,,,,,,,,,,
892,3,0,0,0,0,1,0,0.379085,0.0,1.0,...,1.379085,0.0,0.0,0.0,382.0,0.0,3.0,NaN,0.383721,0.367865
893,3,1,1,0,1,0,0,0.464052,0.0,0.0,...,0.464052,NaN,0.5,1.0,845.0,2.0,2.0,1.0,0.383721,0.867865
894,2,0,0,0,0,1,0,0.209150,0.0,1.0,...,1.209150,0.0,0.0,0.0,382.0,0.0,2.0,NaN,0.349057,0.320755
895,3,0,0,0,0,1,0,0.359477,0.0,0.0,...,1.359477,0.0,0.0,0.0,845.0,0.0,3.0,NaN,0.420000,0.295652
896,3,1,1,1,2,0,0,0.352941,0.0,0.0,...,0.352941,NaN,1.0,3.0,845.0,2.0,1.0,1.0,0.420000,0.795652
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1305,3,0,0,0,0,1,0,0.627451,0.0,0.0,...,1.627451,0.0,0.0,0.0,845.0,0.0,3.0,NaN,0.383721,0.367865
1306,1,1,0,0,0,1,0,0.699346,1.0,0.0,...,1.699346,1.0,0.5,0.0,382.0,0.0,1.0,NaN,0.883721,0.867865
1307,3,0,0,0,0,1,0,0.738562,0.0,0.0,...,1.738562,0.0,0.0,0.0,845.0,0.0,3.0,NaN,0.383721,0.367865


Сохранить в csv

In [11]:
ofe_train_x.to_csv(f"data/train_openfe.csv", index=True)

In [12]:
ofe_test_x.to_csv(f"data/test_openfe.csv", index=True)